# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook simulates data validation with PyDeequ and routes invalid records (missing IDs) to a quarantine Dead Letter Queue (DLQ).

In [ ]:
!pip install pyspark pydeequ
import os
os.environ['SPARK_VERSION'] = '3.3'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pydeequ
from pydeequ.checks import *
from pydeequ.verification import *

# PyDeequ requires a specific Spark setup
spark = SparkSession.builder \
    .appName("Day06_Data_Quality") \
    .config("spark.jars.packages", pydeequ.deequ_maven_coord) \
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord) \
    .getOrCreate()

print("Spark session created with PyDeequ.")

In [ ]:
# Create synthetic data with intentional bad records
data = [
    ("C001", "P123", 32020), # Valid
    (None, "P124", 32020),   # Invalid: missing condition_occurrence_id
    ("C003", None, 32020),   # Invalid: missing person_id
    ("C004", "P126", -1)     # Invalid: negative concept_id (if we check it)
]
df = spark.createDataFrame(data, ["condition_occurrence_id", "person_id", "condition_concept_id"])

print("--- RAW DATA ---")
df.show()

# Setup PyDeequ Verification Suite
check = Check(spark, CheckLevel.Error, "Data Quality Check")

checkResult = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(
        check.hasSize(lambda x: x >= 1) \
        .isComplete("person_id") \
        .isComplete("condition_occurrence_id") \
        .isNonNegative("condition_concept_id")
    ) \
    .run()

print("--- VERIFICATION RESULTS ---")
checkResult_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)
checkResult_df.show(truncate=False)

# Row-level routing for DLQ
valid_df = df.filter(col("person_id").isNotNull() & col("condition_occurrence_id").isNotNull() & (col("condition_concept_id") >= 0))
invalid_df = df.filter(col("person_id").isNull() | col("condition_occurrence_id").isNull() | (col("condition_concept_id") < 0))

print("--- VALID RECORDS (Route to Silver) ---")
valid_df.show()

print("--- INVALID RECORDS (Route to DLQ) ---")
invalid_df.show()
